# Phase 2 — Stationary Distributions and Convergence

Companion notebook to `notes/phase2-dtmc-analysis.md`. Validates the analytical stationary distribution against the numerical solver, computes the spectral gap and mixing time for the headcount chain, and reproduces the convergence heatmap and eigenvalue spectrum.

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.dtmc import MarkovChain

np.set_printoptions(precision=4, suppress=True)
sns.set_style("white")

## 1. Define the headcount chain

In [ ]:
labels = ["Junior", "Mid", "Senior", "Exit"]
P = np.array(
    [
        [0.93, 0.03, 0.00, 0.04],
        [0.00, 0.96, 0.02, 0.02],
        [0.00, 0.00, 0.99, 0.01],
        [0.50, 0.00, 0.00, 0.50],
    ]
)
mc = MarkovChain(P, state_labels=labels)
print("irreducible?", mc.is_irreducible(), "  aperiodic?", mc.is_aperiodic())

## 2. Stationary distribution: analytical vs numerical

The notes derive (by hand) $\pi_J \approx 1/3.39$ and the rest in proportion. Compare with both numerical methods.

In [ ]:
expected_junior = 1.0 / 3.39
pi_hand = expected_junior * np.array([1.0, 0.75, 1.5, 0.14])

table = pd.DataFrame(
    {
        "hand": pi_hand,
        "linear": mc.stationary(method="linear"),
        "eigen": mc.stationary(method="eigen"),
    },
    index=labels,
)
table.round(4)

## 3. Spectrum, spectral gap, and mixing time

In [ ]:
eigs = mc.eigenvalues()
for k, lam in enumerate(eigs, start=1):
    print(f"lambda_{k} = {lam:.4f}   |lambda| = {abs(lam):.4f}")

print(f"\nspectral gap   = {mc.spectral_gap():.4f}")
print(f"t_mix(1/4)     = {mc.mixing_time(eps=0.25)} months")
print(f"t_mix(0.05)    = {mc.mixing_time(eps=0.05)} months")

## 4. Visualise convergence: $P^n$ heatmap

In [ ]:
n_steps = (1, 5, 10, 50, 100)
pi = mc.stationary()
limiting = np.tile(pi, (mc.n_states, 1))

fig, axes = plt.subplots(1, len(n_steps) + 1, figsize=(3.0 * (len(n_steps) + 1), 3.2))
for ax, label_n in zip(axes[:-1], n_steps):
    sns.heatmap(
        mc.n_step(label_n), ax=ax, vmin=0, vmax=1, cmap="rocket_r",
        annot=True, fmt=".2f", cbar=False, square=True,
        xticklabels=labels, yticklabels=labels,
    )
    ax.set_title(f"$P^{{{label_n}}}$")
sns.heatmap(
    limiting, ax=axes[-1], vmin=0, vmax=1, cmap="rocket_r",
    annot=True, fmt=".2f", cbar=False, square=True,
    xticklabels=labels, yticklabels=labels,
)
axes[-1].set_title(r"$\mathbf{1}\pi^\top$")
plt.tight_layout()

## 5. Sensitivity: doubling junior attrition

In [ ]:
P2 = P.copy()
P2[0, 0] = 0.89  # was 0.93
P2[0, 3] = 0.08  # was 0.04
mc_high = MarkovChain(P2, state_labels=labels)

comp = pd.DataFrame(
    {
        "base (p_14=0.04)": mc.stationary(),
        "high churn (p_14=0.08)": mc_high.stationary(),
    },
    index=labels,
).round(4)
comp["% change"] = ((comp.iloc[:, 1] / comp.iloc[:, 0]) - 1).round(3) * 100
comp